# Pre- vs. post-SHINE cohort comparison

Do the pre-SHINE and post-SHINE cohorts *behave the same*? Compares who took part and how they
performed the task -- sample, coverage, demographics, retention, QC flags, trial duration and moves --
for **retained subjects only** (production
cohort, `status == "full data"`, `task_version >= 4.0`; the same selection
`analysis.build_rdms.behavior.behavioral.build_behavioral_rdm()` uses, see `_select_participants()`).
Companion notebooks: `cohort_data_quality.ipynb` (is the data equally good?) and
`cohort_structure.ipynb` (do the cohorts give the same similarity structure?).
Data-prep and figure helpers live in `analysis/behavioral/cohort_comparison.py`; this notebook
loads the data, runs the statistical tests, and renders the figures -- matching this repo's
existing notebook-as-analysis-surface convention (`analysis/pilot/analysis.ipynb`,
`analysis/build_rdms/behavior/aggregation_comparison.ipynb`).

All pairwise comparisons are two-sided Mann-Whitney U tests on per-subject aggregates (one
value per subject, avoiding pseudo-replication across trials), with rank-biserial effect size
*r* = 1 - 2U/(n_A x n_B); |r| >= 0.5 is conventionally large. Categorical comparisons (sex,
screening outcome) use Fisher's exact test. Colours are the validated colorblind-safe
blue/orange pair already used for v1.0/v2.0 in `analysis/pilot/figures.py`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/behavioral/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, spearmanr

from analysis.utils.stats import mwu_effect_size, print_comparison_line
from analysis.utils.equivalence import (
    equivalence_sensitivity, equivalence_sensitivity_risk_difference, equivalence_test_rank_biserial,
    equivalence_test_risk_difference, print_equivalence_line, resolvable_bound,
)
from analysis.utils.plotting import fig_equivalence_forest
from analysis.behavioral.cohort_comparison import (
    COHORTS, COHORT_LABEL, load_retained_cohorts, pair_coverage, subject_trial_stats,
    fig_trial_time_moves, fig_trial_progression, fig_demographics, fig_retention_status,
)


def _print_fisher(label: str, table: list, oddsratio: float, p: float, alpha: float = 0.05) -> None:
    sig = "significant" if p < alpha else "not significant"
    print(f"  {label}: table={table}  odds ratio={oddsratio:.3f}  p={p:.4f} ({sig})")

## 0. Load data

In [2]:
cache_dir = REPO_ROOT / "analysis" / "results" / "parsed_data"  # override if running from a worktree without this cache copied in
cohorts = load_retained_cohorts(cache_dir)
df_participants, df_trials = cohorts["df_participants"], cohorts["df_trials"]
participants, participant_ids, trials_main = cohorts["participants"], cohorts["participant_ids"], cohorts["trials_main"]

for c in COHORTS:
    print(f"{COHORT_LABEL[c]}: {len(participants[c])} retained subjects, {len(trials_main[c])} main trials")

Pre-SHINE: 79 retained subjects, 1738 main trials
Post-SHINE: 76 retained subjects, 1672 main trials


## 1. General statistics

Number of retained subjects, coverage of the full 725-image pair space by at least one SpAM
judgement, and mean +/- SEM number of ratings per (covered) image-pair. (Within-subject reliability
is in `cohort_data_quality.ipynb`.)

In [3]:
coverage = {c: pair_coverage(df_trials, participant_ids[c], c) for c in COHORTS}

general_stats = pd.DataFrame({
    COHORT_LABEL[c]: {
        "N subjects": len(participants[c]),
        "% image-pairs rated >=1x": round(coverage[c]["pct_covered"], 2),
        "Ratings/pair, mean": round(coverage[c]["mean_per_pair"], 3),
        "Ratings/pair, SEM": round(coverage[c]["sem_per_pair"], 4),
    }
    for c in COHORTS
}).T
general_stats

,N subjects,% image-pairs rated >=1x,"Ratings/pair, mean","Ratings/pair, SEM"
Pre-SHINE,79.0,64.57,1.949,0.0027
Post-SHINE,76.0,63.09,1.919,0.0027


## 2. Trial duration and number of moves

Per-trial distributions of trial duration and number of moves (main trials only -- catch
trials use a different `images_per_trial`, see `SpAM_Task/task_config.json`, so mixing them in
would confound the move-count comparison), the per-subject moves-vs-duration relationship, and
how both measures trend over the course of the session (mean +/- SE per trial number, with thin
per-subject lines -- same progression-panel style as `analysis/pilot/figures.py`'s
`fig_trial_duration`/`fig_moves`). Mann-Whitney U tests below use per-subject means.

In [4]:
subj_trial_stats = {c: subject_trial_stats(trials_main[c]) for c in COHORTS}
fig_trial_time_moves(trials_main, subj_trial_stats).show()

In [5]:
fig_trial_progression(trials_main).show()

In [6]:
result_rt = mwu_effect_size(subj_trial_stats["pre"]["rt_s"], subj_trial_stats["post"]["rt_s"], label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"])
result_moves = mwu_effect_size(subj_trial_stats["pre"]["num_moves"], subj_trial_stats["post"]["num_moves"], label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"])
print_comparison_line(result_rt)
print_comparison_line(result_moves)

for c in COHORTS:
    rho, p = spearmanr(subj_trial_stats[c]["num_moves"], subj_trial_stats[c]["rt_s"])
    print(f"  {COHORT_LABEL[c]} moves-vs-duration (per subject): Spearman rho={rho:.3f}  p={p:.4f}")

pd.DataFrame([result_rt, result_moves], index=["Trial duration (s)", "Number of moves"])

  Pre-SHINE vs Post-SHINE: U=3089.0  p=0.7569 (not significant)  r=-0.029
  Pre-SHINE vs Post-SHINE: U=2682.0  p=0.2528 (not significant)  r=0.107
  Pre-SHINE moves-vs-duration (per subject): Spearman rho=0.305  p=0.0062
  Post-SHINE moves-vs-duration (per subject): Spearman rho=0.138  p=0.2334


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
Trial duration (s),Pre-SHINE vs Post-SHINE,79,76,79.1208,78.9177,86.8618,84.2233,22.4874,19.2530,3089.0,0.756853,-0.0290
Number of moves,Pre-SHINE vs Post-SHINE,79,76,24.6364,25.7955,27.0984,27.2967,9.7595,6.4409,2682.0,0.252777,0.1066


## 3. Age and gender distributions

Age (Mann-Whitney U) and sex (Fisher's exact test on the Male/Female counts) distributions for
each cohort's retained subjects.

In [7]:
fig_demographics(participants).show()

In [8]:
result_age = mwu_effect_size(participants["pre"]["age"], participants["post"]["age"], label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"])
print_comparison_line(result_age)

sex_counts = {c: participants[c]["sex"].value_counts() for c in COHORTS}
sex_table = [
    [int(sex_counts["pre"].get("Male", 0)), int(sex_counts["post"].get("Male", 0))],
    [int(sex_counts["pre"].get("Female", 0)), int(sex_counts["post"].get("Female", 0))],
]
odds_ratio, p_sex = fisher_exact(sex_table)
_print_fisher("Sex (Male/Female x cohort)", sex_table, odds_ratio, p_sex)

pd.DataFrame([result_age], index=["Age"])

  Pre-SHINE vs Post-SHINE: U=2899.5  p=0.7148 (not significant)  r=0.034
  Sex (Male/Female x cohort): table=[[42, 38], [37, 37]]  odds ratio=1.105  p=0.8719 (not significant)


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
Age,Pre-SHINE vs Post-SHINE,79,76,29.0,29.0,30.6835,31.7237,8.9364,11.0883,2899.5,0.714755,0.0341


## 4. Additional comparisons

Two comparisons that speak directly to whether the retained pre- and post-SHINE samples are
equivalent, beyond what was asked for above:

- **Screening/retention outcome**: among *all* `task_version >= 4.0` production participants
  (not just retained ones), does the rate of ending up "full data" vs. screened out / flagged
  as a false positive on the full-session recheck differ by cohort? A differential retention
  rate would mean the two retained samples aren't drawn from equivalently-screened pools.
- **QC flag rate**: per-subject rate of the recorded main-trial `qc_flag` (see
  `SpAM_Task/js/utils.js`'s `computeMainQcFlag`), compared between cohorts -- a proxy for
  whether one SHINE variant's images were harder to arrange sensibly.

In [9]:
v4plus = df_participants[pd.to_numeric(df_participants["task_version"], errors="coerce") >= 4.0]
v4plus = v4plus[v4plus["cohort"] == "production"]

fig_retention_status(v4plus).show()

In [10]:
full_data = v4plus["status"] == "full data"
retention_table = [
    [int((full_data & (v4plus["shine_variant"] == "pre")).sum()), int((full_data & (v4plus["shine_variant"] == "post")).sum())],
    [int((~full_data & (v4plus["shine_variant"] == "pre")).sum()), int((~full_data & (v4plus["shine_variant"] == "post")).sum())],
]
odds_ratio, p_retention = fisher_exact(retention_table)
_print_fisher("Retained (full data) vs not, by cohort", retention_table, odds_ratio, p_retention)

  Retained (full data) vs not, by cohort: table=[[79, 76], [22, 18]]  odds ratio=0.850  p=0.7238 (not significant)


In [11]:
subj_qc_rate = {
    c: trials_main[c].groupby("participant_id")["qc_flag"].mean()
    for c in COHORTS
}
result_qc = mwu_effect_size(subj_qc_rate["pre"], subj_qc_rate["post"], label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"])
print_comparison_line(result_qc)
pd.DataFrame([result_qc])

  Pre-SHINE vs Post-SHINE: U=3412.0  p=0.0136 (significant)  r=-0.137


,comparison,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p,r
0,Pre-SHINE vs Post-SHINE,79,76,0.0,0.0,0.0109,0.0036,0.0233,0.0144,3412.0,0.013564,-0.1366


## 5. Equivalence tests

Every "no difference" above is a non-significant Mann-Whitney or Fisher test, which is absence of
evidence rather than evidence of equivalence. This section tests equivalence directly (TOST in its
confidence-interval form, `analysis/utils/equivalence.py`): a comparison is *equivalent* when the 90%
CI of its effect size lies entirely inside +/- a bound.

- **Per-subject outcomes** (age, trial duration, moves): rank-biserial *r* with a subject-level bootstrap CI.
- **Proportions** (sex, retention): risk difference with a Newcombe score CI, in proportion units.

The bound is not chosen by hand. There is no pre-defined smallest effect size of interest for this
project, so every bound here follows one rule: the smallest bound with 80% power when the groups truly do
not differ (a simulation at the observed sample sizes, `analysis/utils/equivalence.py`), rounded up. A
bound tighter than that would fail to show equivalence even for identical groups too often to be
informative. The QC-flag rate is excluded because it is a difference, not a null result.

In [12]:
n_pre, n_post = len(participants["pre"]), len(participants["post"])
rank_sensitivity = equivalence_sensitivity(n_pre, n_post)
rank_bound = resolvable_bound(rank_sensitivity)
print(f"Rank-biserial: 80%-power sensitivity at n=({n_pre}, {n_post}) = {rank_sensitivity:.3f} -> bound +/-{rank_bound:.2f}")

Rank-biserial: 80%-power sensitivity at n=(79, 76) = 0.269 -> bound +/-0.30


In [13]:
rank_outcomes = {
    "Age": (participants["pre"]["age"], participants["post"]["age"]),
    "Trial duration (s)": (subj_trial_stats["pre"]["rt_s"], subj_trial_stats["post"]["rt_s"]),
    "Number of moves": (subj_trial_stats["pre"]["num_moves"], subj_trial_stats["post"]["num_moves"]),
}
equivalence_rank = {
    name: equivalence_test_rank_biserial(
        pre, post, bound=rank_bound, label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"],
    )
    for name, (pre, post) in rank_outcomes.items()
}
for name, result in equivalence_rank.items():
    print(name)
    print_equivalence_line(result)
fig_equivalence_forest(equivalence_rank, x_title="Rank-biserial r (Pre vs Post), 90% CI", title="Equivalence: per-subject outcomes").show()

Age
  Pre-SHINE vs Post-SHINE: effect=0.034  90% CI [-0.116, 0.185]  bound=+/-0.3 (equivalent)
Trial duration (s)
  Pre-SHINE vs Post-SHINE: effect=-0.029  90% CI [-0.181, 0.129]  bound=+/-0.3 (equivalent)
Number of moves
  Pre-SHINE vs Post-SHINE: effect=0.107  90% CI [-0.044, 0.262]  bound=+/-0.3 (equivalent)


In [14]:
male_counts = {c: int(sex_counts[c].get("Male", 0)) for c in COHORTS}
sex_totals = {c: int(sex_counts[c].get("Male", 0) + sex_counts[c].get("Female", 0)) for c in COHORTS}
retained_counts = {c: int((full_data & (v4plus["shine_variant"] == c)).sum()) for c in COHORTS}
eligible_counts = {c: int((v4plus["shine_variant"] == c).sum()) for c in COHORTS}

proportion_counts = {
    "Sex (proportion male)": (male_counts, sex_totals),
    "Retention (proportion retained)": (retained_counts, eligible_counts),
}
equivalence_prop = {}
for name, (successes, totals) in proportion_counts.items():
    pooled = (successes["pre"] + successes["post"]) / (totals["pre"] + totals["post"])
    sensitivity = equivalence_sensitivity_risk_difference(totals["pre"], totals["post"], pooled)
    bound = resolvable_bound(sensitivity)
    print(f"{name}: pooled proportion {pooled:.2f}, 80%-power sensitivity {sensitivity:.3f} -> bound +/-{bound:.2f}")
    equivalence_prop[name] = equivalence_test_risk_difference(
        successes["pre"], totals["pre"], successes["post"], totals["post"],
        bound=bound, label_a=COHORT_LABEL["pre"], label_b=COHORT_LABEL["post"],
    )
    print_equivalence_line(equivalence_prop[name])
fig_equivalence_forest(equivalence_prop, x_title="Risk difference (Pre - Post), 90% CI", title="Equivalence: proportions").show()

Sex (proportion male): pooled proportion 0.52, 80%-power sensitivity 0.232 -> bound +/-0.25
  Pre-SHINE vs Post-SHINE: effect=0.025  90% CI [-0.106, 0.155]  bound=+/-0.25 (equivalent)
Retention (proportion retained): pooled proportion 0.79, 80%-power sensitivity 0.169 -> bound +/-0.20
  Pre-SHINE vs Post-SHINE: effect=-0.026  90% CI [-0.120, 0.070]  bound=+/-0.2 (equivalent)
